<a href="https://colab.research.google.com/github/NaziaAfreen015/CSC791-DLBA/blob/main/Resnet_FGSM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import tensorflow as tf
import torchvision.models as models
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO
import torch
import torch.optim as optim
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
import torchvision

import copy
import math
from typing import List, Tuple
import time


import cv2
import matplotlib.pyplot as plt
import json

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
def get_train_transform(transform_size=224):
  train_transform = transforms.Compose([
      transforms.Resize((transform_size, transform_size)),
      transforms.RandomHorizontalFlip(),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
  ])
  return train_transform

def get_test_transform(transform_size):
  test_transform = transforms.Compose([
      transforms.Resize((transform_size, transform_size)),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
  ])
  return test_transform


In [7]:
def get_train_val_split(dataset='10', val_ratio=0.1, seed=42):
  if dataset == '10':
    full_train_for_indices = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)
  else:
    full_train_for_indices = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=None)

  num_train = len(full_train_for_indices)
  num_val = int(val_ratio * num_train)
  num_train_split = num_train - num_val

  generator = torch.Generator().manual_seed(seed)
  permuted_indices = torch.randperm(num_train, generator=generator).tolist()

  train_indices = permuted_indices[:num_train_split]
  val_indices = permuted_indices[num_train_split:]

  return train_indices, val_indices


In [8]:
# from torchvision.datasets.vision import data
def load_dataset(dataset='10', val_ratio=0.1, seed=42, batch_size=64):
  train_transform = get_train_transform(transform_size=224)
  test_transform = get_test_transform(transform_size=224)

  if dataset == '10':
    train_indices, val_indices = get_train_val_split(dataset=dataset, val_ratio=val_ratio, seed=seed)

    trainset_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=train_transform)
    valset_full = datasets.CIFAR10(root="./data", train=True, download=False, transform=test_transform)

    trainset = Subset(trainset_full, train_indices)
    valset = Subset(valset_full, val_indices)

    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    # Download and load test dataset
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

    print('CIFAR-10 dataset downloaded and loaded successfully.')

  elif dataset == '100':
    train_indices, val_indices = get_train_val_split(dataset=dataset, val_ratio=val_ratio, seed=seed)

    trainset_full = torchvision.datasets.CIFAR100(root='./data', train=True, download=False, transform=train_transform)
    valset_full = datasets.CIFAR100(root="./data", train=True, download=False, transform=test_transform)

    trainset = Subset(trainset_full, train_indices)
    valset = Subset(valset_full, val_indices)

    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    # Download and load test dataset
    testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=test_transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    classes = ('beaver', 'dolphin', 'otter', 'seal', 'whale', 'aquarium fish', 'flatfish', 'ray', 'shark', 'trout', 'orchids', 'poppies', 'roses', 'sunflowers', 'tulips', 'bottles', 'bowls', 'cans', 'cups', 'plates', 'apples', 'mushrooms', 'oranges', 'pears', 'sweet peppers', 'clock', 'computer keyboard', 'lamp', 'telephone', 'television', 'bed', 'chair', 'couch', 'table', 'wardrobe', 'bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach', 'bear', 'leopard', 'lion', 'tiger', 'wolf', 'bridge', 'castle', 'house', 'road', 'skyscraper', 'cloud', 'forest', 'mountain', 'plain', 'sea', 'camel', 'cattle', 'chimpanzee', 'elephant', 'kangaroo', 'fox', 'porcupine', 'possum', 'raccoon', 'skunk', 'crab', 'lobster', 'snail', 'spider', 'worm', 'baby', 'boy', 'girl', 'man', 'woman', 'crocodile', 'dinosaur', 'lizard', 'snake', 'turtle', 'hamster', 'mouse', 'rabbit', 'shrew', 'squirrel', 'maple', 'oak', 'palm', 'pine', 'willow', 'bicycle', 'bus', 'motorcycle', 'pickup truck', 'train', 'lawn-mower', 'rocket', 'streetcar', 'tank', 'tractor')
    print('CIFAR-100 dataset downloaded and loaded successfully.')

  return trainset, trainloader, testset, testloader, valset, valloader, classes


In [9]:
def customize_model(model, num_classes):
  model.fc = nn.Linear(model.fc.in_features, num_classes)
  return model


In [10]:
def get_model(num_classes, path=''):
  if path != '':
    model = models.resnet18(weights=None)
    model = customize_model(model, num_classes)
    print(f'Loading model from path {path}')
    model.load_state_dict(torch.load(path, map_location=device))
  else:
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model = customize_model(model, num_classes)
  model = model.to(device)
  return model

In [11]:
def get_normalized_bounds(mean, std, device):
    """
    Compute the minimum and maximum valid values in normalized space.

    What this function does:
    - Assumes the original image pixels must stay in [0, 1]
    - Converts that valid pixel range into normalized-space bounds
    - Returns tensors shaped for broadcasting over a batch of images
    """
    mean = mean.to(device).view(1, 3, 1, 1)
    std = std.to(device).view(1, 3, 1, 1)

    clamp_min = (0.0 - mean) / std
    clamp_max = (1.0 - mean) / std
    return clamp_min, clamp_max

In [12]:
def fgsm_attack_normalized(image, epsilon, data_grad, std):
    """
    Apply FGSM to normalized images while interpreting epsilon in pixel space.

    What this function does:
    - Takes the gradient w.r.t. the normalized input image
    - Converts the pixel-space epsilon into channel-wise normalized epsilon
    - Adds an FGSM perturbation in normalized space
    - Returns the perturbed normalized image

    Parameters:
    - image: normalized input batch
    - epsilon: perturbation size in raw pixel space, e.g. 8/255
    - data_grad: gradient of loss w.r.t. image
    - std: channel standard deviations used in normalization
    """
    std = std.view(1, 3, 1, 1).to(image.device)
    epsilon_normalized = epsilon / std

    perturbed_image = image + epsilon_normalized * data_grad.sign()
    return perturbed_image

In [13]:

def generate_fgsm_examples(model, testloader, epsilon, device, CIFAR_MEAN, CIFAR_STD):
    """
    Generate FGSM adversarial examples for the test set.

    What this function does:
    - Runs the model on clean test images
    - Computes gradients of loss w.r.t. the input images
    - Creates FGSM adversarial examples using epsilon defined in pixel space
    - Clamps the perturbed images to valid normalized bounds
    - Evaluates the model on both clean and adversarial images
    - Stores results for later analysis or visualization

    Returns:
    - adv_examples: adversarial image batches
    - clean_examples: original image batches
    - labels_list: true labels
    - clean_preds_list: predictions on clean images
    - adv_preds_list: predictions on adversarial images
    - clean_acc: clean accuracy
    - adv_acc: adversarial accuracy
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()

    mean = CIFAR_MEAN.to(device)
    std = CIFAR_STD.to(device)
    clamp_min, clamp_max = get_normalized_bounds(mean, std, device)

    correct_clean = 0
    correct_adv = 0
    total = 0

    batch = 0
    examples_10 = [] # 10: 1 = correct clean pred, 0 = incorrect adv pred
    examples_01 = []
    examples_000 = [] # 000: 0 = incorrect clean pred, 0 = incorrect adv pred, 0 = clean != adv
    examples_001 = [] # 000: 0 = incorrect clean pred, 0 = incorrect adv pred, 1 = clean == adv
    examples_11 = []

    examples_10_count = 0
    examples_01_count = 0
    examples_000_count = 0
    examples_001_count = 0
    examples_11_count = 0


    for images, labels in testloader:

        images = images.to(device)
        labels = labels.to(device)

        images.requires_grad = True

        # Forward pass on clean images
        outputs = model(images)
        loss = criterion(outputs, labels)

        _, clean_preds = torch.max(outputs, 1)
        correct_clean_this_batch = (clean_preds == labels).sum().item()
        correct_clean += correct_clean_this_batch

        model.zero_grad()
        loss.backward()

        data_grad = images.grad.detach()

        # Create adversarial images in normalized space,
        # but with epsilon interpreted in raw pixel space
        adv_images = fgsm_attack_normalized(images, epsilon, data_grad, std)

        # Keep adversarial images within valid normalized image range
        adv_images = torch.max(torch.min(adv_images, clamp_max), clamp_min)
        adv_images = adv_images.detach()

        # Forward pass on adversarial images
        adv_outputs = model(adv_images)
        _, adv_preds = torch.max(adv_outputs, 1)
        correct_adv_this_batch = (adv_preds == labels).sum().item()
        correct_adv += correct_adv_this_batch

        total += labels.size(0)

        # clean_examples.append(images.detach().cpu())
        # adv_examples.append(adv_images.cpu())
        # labels_list.append(labels.detach().cpu())
        # clean_preds_list.append(clean_preds.detach().cpu())
        # adv_preds_list.append(adv_preds.detach().cpu())

        examples_10_mask = (clean_preds == labels) & (adv_preds != labels)
        examples_10_indices = examples_10_mask.nonzero(as_tuple=True)[0]
        examples_10_count += len(examples_10_indices)
        examples_10_indices = examples_10_indices[:1]
        if len(examples_10) == 0:
          for idx in examples_10_indices:
              examples_10.append({
                  "clean_image": images[idx].detach().cpu(),
                  "adv_image": adv_images[idx].detach().cpu(),
                  "label": labels[idx].item(),
                  "clean_pred": clean_preds[idx].item(),
                  "adv_pred": adv_preds[idx].item(),
              })

        examples_01_mask = (clean_preds != labels) & (adv_preds == labels)
        examples_01_indices = examples_01_mask.nonzero(as_tuple=True)[0]
        examples_01_count += len(examples_01_indices)
        examples_01_indices = examples_01_indices[:1]
        if len(examples_01) == 0:
          for idx in examples_01_indices:
            examples_01.append({
                "clean_image": images[idx].detach().cpu(),
                "adv_image": adv_images[idx].detach().cpu(),
                "label": labels[idx].item(),
                "clean_pred": clean_preds[idx].item(),
                "adv_pred": adv_preds[idx].item(),
            })

        examples_000_mask = (clean_preds != labels) & (adv_preds != labels) & (clean_preds != adv_preds)
        examples_000_indices = examples_000_mask.nonzero(as_tuple=True)[0]
        examples_000_count += len(examples_000_indices)
        examples_000_indices = examples_000_indices[:1]
        if len(examples_000) == 0:
          for idx in examples_000_indices:
              examples_000.append({
                  "clean_image": images[idx].detach().cpu(),
                  "adv_image": adv_images[idx].detach().cpu(),
                  "label": labels[idx].item(),
                  "clean_pred": clean_preds[idx].item(),
                  "adv_pred": adv_preds[idx].item(),
              })

        examples_001_mask = (clean_preds != labels) & (adv_preds != labels) & (clean_preds == adv_preds)
        examples_001_indices = examples_001_mask.nonzero(as_tuple=True)[0]
        examples_001_count += len(examples_001_indices)
        examples_001_indices = examples_001_indices[:1]
        if len(examples_001) == 0:
          for idx in examples_001_indices:
              examples_001.append({
                  "clean_image": images[idx].detach().cpu(),
                  "adv_image": adv_images[idx].detach().cpu(),
                  "label": labels[idx].item(),
                  "clean_pred": clean_preds[idx].item(),
                  "adv_pred": adv_preds[idx].item(),
              })

        examples_11_mask = (clean_preds == labels) & (adv_preds == labels)
        examples_11_indices = examples_11_mask.nonzero(as_tuple=True)[0]
        examples_11_count += len(examples_11_indices)
        examples_11_indices = examples_11_indices[:1]
        if len(examples_11) == 0:
          for idx in examples_11_indices:
              examples_11.append({
                  "clean_image": images[idx].detach().cpu(),
                  "adv_image": adv_images[idx].detach().cpu(),
                  "label": labels[idx].item(),
                  "clean_pred": clean_preds[idx].item(),
                  "adv_pred": adv_preds[idx].item(),
              })


        # print(f"batch {batch}\tcorrect clean {correct_clean_this_batch} correct adv {correct_adv_this_batch} total {labels.size(0)}")
        if batch % 20 == 0:
          print(f'Batch {batch} done')
        batch += 1

    clean_acc = correct_clean / total
    adv_acc = correct_adv / total

    return (
        examples_10, examples_01, examples_000, examples_001, examples_11, examples_10_count, examples_01_count, examples_000_count, examples_001_count, examples_11_count, clean_acc, adv_acc
    )

In [23]:
def measure_fgsm(model, testloader, epsilons, device, CIFAR_MEAN, CIFAR_STD):
  # epsilon = 8 / 255  # standard FGSM budget in pixel space


  clean_accuracy = -1
  adv_acc_list = []

  for epsilon in epsilons:
    print(f'epsilon: {epsilon:.2f}')
    examples_10, examples_01, examples_000, examples_001, examples_11, examples_10_count, examples_01_count, examples_000_count, examples_001_count, examples_11_count, clean_acc, adv_acc = generate_fgsm_examples(
        model=model,
        testloader=testloader,
        epsilon=epsilon,
        device=device,
        CIFAR_MEAN=CIFAR_MEAN,
        CIFAR_STD=CIFAR_STD
    )
    if clean_accuracy == -1:
      clean_accuracy = clean_acc
    # ["Dense", "0.0735", "0.1234", "0.1426", "0.1457"],
    adv_acc_list.append(adv_acc)
  return clean_accuracy, adv_acc_list







    # common_name = '/content/drive/MyDrive/DLBA_Project/FGSM_Fig_report/FGSM_prune_finetune'
    # file_name = common_name+str(sparsity)+'_VGG_100.txt'

    # with open(file_name, 'a') as f:
    #   f.write("Epsilon: "+str(epsilon)+"\n")
    #   f.write("examples_10_count: "+str(examples_10_count)+"\n")
    #   f.write("examples_01_count: "+str(examples_01_count)+"\n")
    #   f.write("examples_000_count: "+str(examples_000_count)+"\n")
    #   f.write("examples_001_count: "+str(examples_001_count)+"\n")
    #   f.write("examples_11_count: "+str(examples_11_count)+'\n')
    #   f.write("Clean accuracy: "+str(clean_acc)+"\n")
    #   f.write("FGSM accuracy (epsilon="+str(epsilon)+"): "+str(adv_acc)+'\n\n\n\n')

    # epsilon_cap = f"{epsilon:.2f}"
    # for index,fig_list in enumerate([examples_10, examples_01, examples_000, examples_001, examples_11]):
    #   name_list = ['10', '01', '000', '001', '11']
    #   if len(fig_list) == 0:
    #     continue

    #   plt = plot_clean_adv_pair(fig_list[0], CIFAR_MEAN, CIFAR_STD)
    #   save_path = common_name+str(sparsity)+'_VGG_100_'+str(name_list[index])+"_"+str(epsilon_cap)+".jpg"
    #   # save_path = '/content/drive/MyDrive/DLBA_Project/FGSM_Fig_report/FGSM_prune_finetune_VGG_'
    #   plt.savefig(save_path,  format='jpg', dpi=300)



## Driver Code

 ```{
    "title": "Clean Accuracy by Model:",
    "headers": ["Model Type", "Clean Accuracy"],
    "data": [
      ["Dense", "0.9155"],
      ["Pruned 20%", "0.9185"],
      ["Pruned 40%", "0.9186"],
      ["Pruned 60%", "0.9198"],
      ["Pruned 80%", "0.9161"]
    ]
  },
  {
    "title": "Adversarial accuracy (same-model adversarial examples):",
    "headers": ["Model Variant", "0.0078", "0.0156", "0.0313", "0.0627"],
    "data": [
      ["Dense", "0.0735", "0.1234", "0.1426", "0.1457"],
      ["Pruned 20%", "0.0731", "0.1215", "0.1424", "0.1466"],
      ["Pruned 40%", "0.0706", "0.1227", "0.1399", "0.1443"],
      ["Pruned 60%", "0.0730", "0.1204", "0.1396", "0.1430"],
      ["Pruned 80%", "0.0727", "0.1207", "0.1389", "0.1387"]
    ]
  }
  ```

In [20]:
# CIFAR test normalization used by your loader
CIFAR_MEAN = torch.tensor([0.485, 0.456, 0.406])
CIFAR_STD = torch.tensor([0.229, 0.224, 0.225])


In [24]:
sparsity_list =["Dense", "Pruned 20%", "Pruned 40%", "Pruned 60%", "Pruned 80%" ]
model_sparsities = [0,20,40,60,80]

epsilons = [2/255, 4/255, 8/255, 16/255]

for num_class in ['10', '100']:
  all_result = [
    {"title": "Clean Accuracy by Model:",
   "headers": ["Model Type", "Clean Accuracy"],
   "data": [
    #  ["Dense", ],
    #  ["Pruned 20%", ],
    #  ["Pruned 40%", ],
    #  ["Pruned 60%", ],
    #  ["Pruned 80%", ]
   ]},
     {
    "title": "Adversarial accuracy (same-model adversarial examples):",
    "headers": ["Model Variant", "0.0078", "0.0156", "0.0313", "0.0627"],
    "data": [
      # ["Dense", "0.0735", "0.1234", "0.1426", "0.1457"],
      # ["Pruned 20%", "0.0731", "0.1215", "0.1424", "0.1466"],
      # ["Pruned 40%", "0.0706", "0.1227", "0.1399", "0.1443"],
      # ["Pruned 60%", "0.0730", "0.1204", "0.1396", "0.1430"],
      # ["Pruned 80%", "0.0727", "0.1207", "0.1389", "0.1387"]
    ]
  }
    ]

  trainset, trainloader, testset, testloader, valset, valloader, classes = load_dataset(dataset = num_class)
  model_load_paths = [
      f'/content/drive/MyDrive/DLBA_Project/Models/finetuned_resnet18_cifar{num_class}_base_model.pth',
      f'/content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar{num_class}_20.pth',
      f'/content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar{num_class}_40.pth',
      f'/content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar{num_class}_60.pth',
      f'/content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar{num_class}_80.pth'
  ]
  for idx, path in enumerate(model_load_paths):
    sparsity = model_sparsities[idx]
    load_path = path
    model = get_model(num_classes=int(num_class), path=load_path)
    clean_acc, adv_acc_list = measure_fgsm(model, testloader, epsilons, device, CIFAR_MEAN, CIFAR_STD)
    all_result[1]['data'].append([sparsity_list[idx],adv_acc_list])
    all_result[0]["data"].append([sparsity_list[idx],clean_acc])

  with open(f'/content/drive/MyDrive/DLBA_Project/Resnet_Result/Resnet18_T1_C{num_class}_result.json', 'a') as f:
    json.dump(all_result, f)

CIFAR-10 dataset downloaded and loaded successfully.
Loading model from path /content/drive/MyDrive/DLBA_Project/Models/finetuned_resnet18_cifar10_base_model.pth
epsilon: 0.01
Batch 0 done
Batch 20 done
Batch 40 done
Batch 60 done
Batch 80 done
Batch 100 done
Batch 120 done
Batch 140 done
epsilon: 0.02
Batch 0 done
Batch 20 done
Batch 40 done
Batch 60 done
Batch 80 done
Batch 100 done
Batch 120 done
Batch 140 done
epsilon: 0.03
Batch 0 done
Batch 20 done
Batch 40 done
Batch 60 done
Batch 80 done
Batch 100 done
Batch 120 done
Batch 140 done
epsilon: 0.06
Batch 0 done
Batch 20 done
Batch 40 done
Batch 60 done
Batch 80 done
Batch 100 done
Batch 120 done
Batch 140 done
Loading model from path /content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar10_20.pth
epsilon: 0.01
Batch 0 done
Batch 20 done
Batch 40 done
Batch 60 done
Batch 80 done
Batch 100 done
Batch 120 done
Batch 140 done
epsilon: 0.02
Batch 0 done
Batch 20 done
Batch 40 done
Batch 60 done
Batch 80 done
Batch 100 done
Ba

100%|██████████| 169M/169M [00:18<00:00, 9.04MB/s]


CIFAR-100 dataset downloaded and loaded successfully.
Loading model from path /content/drive/MyDrive/DLBA_Project/Models/finetuned_resnet18_cifar100_base_model.pth
epsilon: 0.01
Batch 0 done
Batch 20 done
Batch 40 done
Batch 60 done
Batch 80 done
Batch 100 done
Batch 120 done
Batch 140 done
epsilon: 0.02
Batch 0 done
Batch 20 done
Batch 40 done
Batch 60 done
Batch 80 done
Batch 100 done
Batch 120 done
Batch 140 done
epsilon: 0.03
Batch 0 done
Batch 20 done
Batch 40 done
Batch 60 done
Batch 80 done
Batch 100 done
Batch 120 done
Batch 140 done
epsilon: 0.06
Batch 0 done
Batch 20 done
Batch 40 done
Batch 60 done
Batch 80 done
Batch 100 done
Batch 120 done
Batch 140 done
Loading model from path /content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar100_20.pth
epsilon: 0.01
Batch 0 done
Batch 20 done
Batch 40 done
Batch 60 done
Batch 80 done
Batch 100 done
Batch 120 done
Batch 140 done
epsilon: 0.02
Batch 0 done
Batch 20 done
Batch 40 done
Batch 60 done
Batch 80 done
Batch 100 done

# ASR

In [14]:
def evaluate_fgsm_on_initially_correct(
    model,
    testloader,
    epsilon,
    device,
    mean,
    std,
    save_per_batch=0
):
    """
    What this does:
    - Runs the model on clean test images
    - Keeps only the images that are correctly classified on clean data
    - Generates FGSM adversarial examples only for those initially-correct images
    - Runs the model on those adversarial images
    - Measures how many of those originally-correct images become wrong

    Returns:
    - clean_accuracy: accuracy on the full test set
    - adv_accuracy_on_initially_correct: accuracy on the initially-correct subset after attack
    - attack_success_rate: fraction of initially-correct examples that become wrong
    - num_initially_correct: number of clean-correct examples used for attack
    - num_attack_success: number of those that were flipped
    - saved_examples: optional list of successful examples
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()

    mean = mean.to(device)
    std = std.to(device)
    clamp_min, clamp_max = get_normalized_bounds(mean, std, device)

    total = 0
    clean_correct_total = 0

    initially_correct_total = 0
    adv_correct_on_initially_correct = 0
    attack_success_total = 0

    saved_examples = []

    for batch_idx, (images, labels) in enumerate(testloader):
        if batch_idx % 20 == 0:
          print(f"[FGSM initially-correct eval] Batch {batch_idx}")

        images = images.to(device)
        labels = labels.to(device)

        batch_size = labels.size(0)
        total += batch_size

        # Step 1: clean inference on whole batch
        with torch.no_grad():
            clean_outputs = model(images)
            clean_preds = clean_outputs.argmax(dim=1)

        clean_correct_mask = (clean_preds == labels)
        clean_correct_total += clean_correct_mask.sum().item()

        # Step 2: keep only initially-correct examples
        if clean_correct_mask.sum().item() == 0:
            del images, labels, clean_outputs, clean_preds
            continue

        correct_images = images[clean_correct_mask].detach().clone()
        correct_labels = labels[clean_correct_mask].detach().clone()
        correct_clean_preds = clean_preds[clean_correct_mask].detach().clone()

        initially_correct_total += correct_labels.size(0)

        # Step 3: generate FGSM on only the initially-correct subset
        correct_images.requires_grad_(True)

        model.zero_grad(set_to_none=True)
        outputs_for_grad = model(correct_images)
        loss = criterion(outputs_for_grad, correct_labels)
        loss.backward()

        data_grad = correct_images.grad.detach()

        adv_images = fgsm_attack_normalized(correct_images, epsilon, data_grad, std)
        adv_images = torch.max(torch.min(adv_images, clamp_max), clamp_min).detach()

        # Step 4: run model on adversarial images
        with torch.no_grad():
            adv_outputs = model(adv_images)
            adv_preds = adv_outputs.argmax(dim=1)

        adv_correct_on_initially_correct += (adv_preds == correct_labels).sum().item()

        attack_success_mask = (adv_preds != correct_labels)
        attack_success_total += attack_success_mask.sum().item()

        if save_per_batch > 0:
            idxs = attack_success_mask.nonzero(as_tuple=True)[0][:save_per_batch]
            for idx in idxs:
                saved_examples.append({
                    "clean_image": correct_images[idx].detach().cpu(),
                    "adv_image": adv_images[idx].detach().cpu(),
                    "label": correct_labels[idx].item(),
                    "clean_pred": correct_clean_preds[idx].item(),
                    "adv_pred": adv_preds[idx].item(),
                })

        del images, labels
        del clean_outputs, clean_preds
        del correct_images, correct_labels, correct_clean_preds
        del outputs_for_grad, loss, data_grad
        del adv_images, adv_outputs, adv_preds

    clean_accuracy = clean_correct_total / total if total > 0 else 0.0
    adv_accuracy_on_initially_correct = (
        adv_correct_on_initially_correct / initially_correct_total
        if initially_correct_total > 0 else 0.0
    )
    attack_success_rate = (
        attack_success_total / initially_correct_total
        if initially_correct_total > 0 else 0.0
    )

    return {
        "clean_accuracy": clean_accuracy,
        "adv_accuracy_on_initially_correct": adv_accuracy_on_initially_correct,
        "attack_success_rate": attack_success_rate,
        "num_initially_correct": initially_correct_total,
        "num_attack_success": attack_success_total,
        "saved_examples": saved_examples,
    }

In [15]:
# Same normalization as your CIFAR test loader
CIFAR_MEAN = torch.tensor([0.485, 0.456, 0.406])
CIFAR_STD = torch.tensor([0.229, 0.224, 0.225])

In [16]:
sparsity_list =["Dense", "Pruned 20%", "Pruned 40%", "Pruned 60%", "Pruned 80%" ]
model_sparsities = [0,20,40,60,80]

epsilons = [2/255, 4/255, 8/255, 16/255]

for num_class in ['10', '100']:
  all_result = [
      {
          "title": "Attack success rate:",
          "headers": ["Model Variant", "0.0078", "0.0156", "0.0313", "0.0627"],
          "data": []
      },
      {
          "title": "Adversarial accuracy on initially-correct subset:",
          "headers": ["Model Variant", "0.0078", "0.0156", "0.0313", "0.0627"],
          "data": []
      },
      {
          "title": "Initially correct examples:",
          "headers": ["Model Variant", "Initially correct examples"],
          "data": []
      },
      {
        "title": "Successful attacks:",
        "headers": ["Model Variant", "0.0078", "0.0156", "0.0313", "0.0627"],
        "data": []
      }
    ]

  trainset, trainloader, testset, testloader, valset, valloader, classes = load_dataset(dataset = num_class)
  model_load_paths = [
      f'/content/drive/MyDrive/DLBA_Project/Models/finetuned_resnet18_cifar{num_class}_base_model.pth',
      f'/content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar{num_class}_20.pth',
      f'/content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar{num_class}_40.pth',
      f'/content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar{num_class}_60.pth',
      f'/content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar{num_class}_80.pth'
  ]
  for idx, path in enumerate(model_load_paths):
    sparsity = model_sparsities[idx]
    load_path = path
    model = get_model(num_classes=int(num_class), path=load_path)
    asr_list = [sparsity_list[idx]]
    adv_acc_list = []
    correct_list = []
    suc_attack_list = []
    for epsilon in epsilons:
      results = evaluate_fgsm_on_initially_correct(
        model=model,
        testloader=testloader,
        epsilon=epsilon,
        device=device,
        mean=CIFAR_MEAN,
        std=CIFAR_STD,
        save_per_batch=0
        )
      asr_list.append(results['attack_success_rate'])
      adv_acc_list.append(results['adv_accuracy_on_initially_correct'])
      correct_list.append(results['num_initially_correct'])
      suc_attack_list.append(results['num_attack_success'])
    all_result[0]["data"].append(asr_list)
    all_result[1]["data"].append(adv_acc_list)
    all_result[2]["data"].append(correct_list)
    all_result[3]["data"].append(suc_attack_list)


  with open(f'/content/drive/MyDrive/DLBA_Project/Resnet_Result/Resnet18_T1_C{num_class}_result.json', 'a') as f:
    json.dump(all_result, f)

100%|██████████| 170M/170M [00:13<00:00, 12.7MB/s]


CIFAR-10 dataset downloaded and loaded successfully.
Loading model from path /content/drive/MyDrive/DLBA_Project/Models/finetuned_resnet18_cifar10_base_model.pth
[FGSM initially-correct eval] Batch 0
[FGSM initially-correct eval] Batch 20
[FGSM initially-correct eval] Batch 40
[FGSM initially-correct eval] Batch 60
[FGSM initially-correct eval] Batch 80
[FGSM initially-correct eval] Batch 100
[FGSM initially-correct eval] Batch 120
[FGSM initially-correct eval] Batch 140
[FGSM initially-correct eval] Batch 0
[FGSM initially-correct eval] Batch 20
[FGSM initially-correct eval] Batch 40
[FGSM initially-correct eval] Batch 60
[FGSM initially-correct eval] Batch 80
[FGSM initially-correct eval] Batch 100
[FGSM initially-correct eval] Batch 120
[FGSM initially-correct eval] Batch 140
[FGSM initially-correct eval] Batch 0
[FGSM initially-correct eval] Batch 20
[FGSM initially-correct eval] Batch 40
[FGSM initially-correct eval] Batch 60
[FGSM initially-correct eval] Batch 80
[FGSM initially-

100%|██████████| 169M/169M [00:13<00:00, 12.9MB/s]


CIFAR-100 dataset downloaded and loaded successfully.
Loading model from path /content/drive/MyDrive/DLBA_Project/Models/finetuned_resnet18_cifar100_base_model.pth
[FGSM initially-correct eval] Batch 0
[FGSM initially-correct eval] Batch 20
[FGSM initially-correct eval] Batch 40
[FGSM initially-correct eval] Batch 60
[FGSM initially-correct eval] Batch 80
[FGSM initially-correct eval] Batch 100
[FGSM initially-correct eval] Batch 120
[FGSM initially-correct eval] Batch 140
[FGSM initially-correct eval] Batch 0
[FGSM initially-correct eval] Batch 20
[FGSM initially-correct eval] Batch 40
[FGSM initially-correct eval] Batch 60
[FGSM initially-correct eval] Batch 80
[FGSM initially-correct eval] Batch 100
[FGSM initially-correct eval] Batch 120
[FGSM initially-correct eval] Batch 140
[FGSM initially-correct eval] Batch 0
[FGSM initially-correct eval] Batch 20
[FGSM initially-correct eval] Batch 40
[FGSM initially-correct eval] Batch 60
[FGSM initially-correct eval] Batch 80
[FGSM initiall

# FGSM Transfer

In [17]:
def init_transfer_stats():
    return {
        "total": 0,
        "source_clean_correct": 0,
        "target_clean_correct": 0,
        "source_adv_correct": 0,
        "target_adv_correct": 0,
        "target_initially_correct": 0,
        "target_transfer_success": 0,
    }


def finalize_transfer_stats(stats):
    total = stats["total"]
    return {
        "source_clean_acc": stats["source_clean_correct"] / total,
        "target_clean_acc": stats["target_clean_correct"] / total,
        "source_adv_acc": stats["source_adv_correct"] / total,
        "target_adv_acc": stats["target_adv_correct"] / total,
        "target_transfer_fooling_rate": (
            stats["target_transfer_success"] / stats["target_initially_correct"]
            if stats["target_initially_correct"] > 0 else 0.0
        ),
    }


def update_transfer_stats(
    stats,
    labels,
    source_clean_preds,
    target_clean_preds,
    source_adv_preds,
    target_adv_preds
):
    batch_size = labels.size(0)
    stats["total"] += batch_size

    stats["source_clean_correct"] += (source_clean_preds == labels).sum().item()
    stats["target_clean_correct"] += (target_clean_preds == labels).sum().item()
    stats["source_adv_correct"] += (source_adv_preds == labels).sum().item()
    stats["target_adv_correct"] += (target_adv_preds == labels).sum().item()

    target_correct_mask = (target_clean_preds == labels)
    target_transfer_success_mask = target_correct_mask & (target_adv_preds != labels)

    stats["target_initially_correct"] += target_correct_mask.sum().item()
    stats["target_transfer_success"] += target_transfer_success_mask.sum().item()

In [18]:
def evaluate_dense_to_many_pruned_streaming(
    dense_model,
    pruned_models_dict,
    testloader,
    epsilon,
    device,
    mean,
    std
):
    """
    Generates dense-crafted FGSM once per batch, then evaluates that same batch
    on all pruned target models before moving on.

    pruned_models_dict:
        {
            sparsity1: pruned_model1,
            sparsity2: pruned_model2,
            ...
        }
    """
    dense_model.eval()
    for model in pruned_models_dict.values():
        model.eval()

    criterion = nn.CrossEntropyLoss()
    mean = mean.to(device)
    std = std.to(device)
    clamp_min, clamp_max = get_normalized_bounds(mean, std, device)

    stats_dict = {sparsity: init_transfer_stats() for sparsity in pruned_models_dict}

    for batch_idx, (images, labels) in enumerate(testloader):
        if batch_idx % 20 == 0:
          print(f"[Dense->Many] Batch {batch_idx}")

        images = images.to(device)
        labels = labels.to(device)

        # Dense clean + gradient
        images_for_grad = images.detach().clone().requires_grad_(True)
        dense_model.zero_grad(set_to_none=True)

        dense_clean_outputs = dense_model(images_for_grad)
        dense_clean_preds = dense_clean_outputs.argmax(dim=1)

        loss = criterion(dense_clean_outputs, labels)
        loss.backward()

        data_grad = images_for_grad.grad.detach()
        adv_images = fgsm_attack_normalized(images_for_grad, epsilon, data_grad, std)
        adv_images = torch.max(torch.min(adv_images, clamp_max), clamp_min).detach()

        with torch.no_grad():
            dense_adv_outputs = dense_model(adv_images)
            dense_adv_preds = dense_adv_outputs.argmax(dim=1)

            for sparsity, pruned_model in pruned_models_dict.items():
                pruned_clean_outputs = pruned_model(images)
                pruned_adv_outputs = pruned_model(adv_images)

                pruned_clean_preds = pruned_clean_outputs.argmax(dim=1)
                pruned_adv_preds = pruned_adv_outputs.argmax(dim=1)

                update_transfer_stats(
                    stats_dict[sparsity],
                    labels,
                    dense_clean_preds,
                    pruned_clean_preds,
                    dense_adv_preds,
                    pruned_adv_preds
                )

        del images, labels, images_for_grad
        del dense_clean_outputs, dense_adv_outputs, dense_clean_preds, dense_adv_preds
        del loss, data_grad, adv_images

    return {s: finalize_transfer_stats(stats) for s, stats in stats_dict.items()}

In [19]:
def evaluate_transfer_attack(
    source_model,
    target_model,
    testloader,
    epsilon,
    device,
    mean,
    std,
    save_per_batch=0
):
    """
    Generates FGSM examples on source_model and immediately evaluates on target_model.

    Use this when source_model is changing and you do not want caching.
    """
    source_model.eval()
    target_model.eval()
    criterion = nn.CrossEntropyLoss()

    mean = mean.to(device)
    std = std.to(device)
    clamp_min, clamp_max = get_normalized_bounds(mean, std, device)

    total = 0
    source_clean_correct = 0
    target_clean_correct = 0
    source_adv_correct = 0
    target_adv_correct = 0

    target_initially_correct = 0
    target_transfer_success = 0

    saved_examples = []

    for batch_idx, (images, labels) in enumerate(testloader):
        if batch_idx % 20 == 0:
          print(f"[Direct transfer] Batch {batch_idx}")

        images = images.to(device)
        labels = labels.to(device)

        # Clean target predictions do NOT need gradients
        with torch.no_grad():
            target_clean_outputs = target_model(images)
            target_clean_preds = target_clean_outputs.argmax(dim=1)

        # Source branch needs gradients
        images_for_grad = images.detach().clone().requires_grad_(True)

        source_model.zero_grad(set_to_none=True)

        source_clean_outputs = source_model(images_for_grad)
        source_clean_preds = source_clean_outputs.argmax(dim=1)

        loss = criterion(source_clean_outputs, labels)
        loss.backward()

        data_grad = images_for_grad.grad.detach()

        adv_images = fgsm_attack_normalized(images_for_grad, epsilon, data_grad, std)
        adv_images = torch.max(torch.min(adv_images, clamp_max), clamp_min).detach()

        with torch.no_grad():
            source_adv_outputs = source_model(adv_images)
            target_adv_outputs = target_model(adv_images)

            source_adv_preds = source_adv_outputs.argmax(dim=1)
            target_adv_preds = target_adv_outputs.argmax(dim=1)

        batch_size = labels.size(0)
        total += batch_size

        source_clean_correct += (source_clean_preds == labels).sum().item()
        target_clean_correct += (target_clean_preds == labels).sum().item()
        source_adv_correct += (source_adv_preds == labels).sum().item()
        target_adv_correct += (target_adv_preds == labels).sum().item()

        target_correct_mask = (target_clean_preds == labels)
        target_transfer_success_mask = target_correct_mask & (target_adv_preds != labels)

        target_initially_correct += target_correct_mask.sum().item()
        target_transfer_success += target_transfer_success_mask.sum().item()

        if save_per_batch > 0:
            idxs = target_transfer_success_mask.nonzero(as_tuple=True)[0][:save_per_batch]
            for idx in idxs:
                saved_examples.append({
                    "clean_image": images[idx].detach().cpu(),
                    "adv_image": adv_images[idx].detach().cpu(),
                    "label": labels[idx].item(),
                    "source_clean_pred": source_clean_preds[idx].item(),
                    "source_adv_pred": source_adv_preds[idx].item(),
                    "target_clean_pred": target_clean_preds[idx].item(),
                    "target_adv_pred": target_adv_preds[idx].item(),
                })

        del images, labels, images_for_grad
        del source_clean_outputs, target_clean_outputs
        del source_adv_outputs, target_adv_outputs
        del source_clean_preds, target_clean_preds
        del source_adv_preds, target_adv_preds
        del loss, data_grad, adv_images

    source_clean_acc = source_clean_correct / total
    target_clean_acc = target_clean_correct / total
    source_adv_acc = source_adv_correct / total
    target_adv_acc = target_adv_correct / total

    target_transfer_fooling_rate = (
        target_transfer_success / target_initially_correct
        if target_initially_correct > 0 else 0.0
    )

    return {
        "source_clean_acc": source_clean_acc,
        "target_clean_acc": target_clean_acc,
        "source_adv_acc": source_adv_acc,
        "target_adv_acc": target_adv_acc,
        "target_transfer_fooling_rate": target_transfer_fooling_rate,
        "saved_examples": saved_examples,
    }


In [20]:
# Same normalization as your CIFAR test loader
CIFAR_MEAN = torch.tensor([0.485, 0.456, 0.406])
CIFAR_STD = torch.tensor([0.229, 0.224, 0.225])

In [ ]:
num_classes = 100
dataset = '100'

In [26]:
model_sparsities = [0,20,40,60,80]
epsilons = [2/255, 4/255, 8/255, 16/255]
for num_class in ['100']:
  model_load_paths = [
        f'/content/drive/MyDrive/DLBA_Project/Models/finetuned_resnet18_cifar{num_class}_base_model.pth',
        f'/content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar{num_class}_20.pth',
        f'/content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar{num_class}_40.pth',
        f'/content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar{num_class}_60.pth',
        f'/content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar{num_class}_80.pth'
    ]
  all_result = [
      {
       "title": "Transferability: Dense → Pruned:",
    "headers": ["Pruned Sparsity", "0.0078", "0.0156", "0.0313", "0.0627"],
    "data": [

    ]
  },
  {
    "title": "Transferability: Pruned → Dense:",
    "headers": ["Pruned Sparsity", "0.0078", "0.0156", "0.0313", "0.0627"],
    "data": [

    ]
  },
  {
    "title": "Target adversarial accuracy under transfer: Dense → Pruned:",
    "headers": ["Pruned Sparsity", "0.0078", "0.0156", "0.0313", "0.0627"],
    "data": [

    ]
  },
  {
    "title": "Target adversarial accuracy under transfer: Pruned → Dense:",
    "headers": ["Pruned Sparsity", "0.0078", "0.0156", "0.0313", "0.0627"],
    "data": [

    ]
  }

  ]
  trainset, trainloader, testset, testloader, valset, valloader, classes = load_dataset(dataset = num_class)
  dense_model = get_model(num_classes=int(num_class), path=model_load_paths[0])

  transfer_d_p_lst = [
      ["20%"],
      ["40%"],
      ["60%"],
      ["80%"]
    ]

  transfer_p_d_lst = [
      ["20%"],
      ["40%"],
      ["60%"],
      ["80%"]
    ]
  adv_acc_d_p_lst = [
      ["20%"],
      ["40%"],
      ["60%"],
      ["80%"]
    ]
  adv_acc_p_d_lst = [
      ["20%"],
      ["40%"],
      ["60%"],
      ["80%"]
    ]

  for epsilon in epsilons:
    print(f"\n================ Epsilon: {epsilon} ================\n")
    pruned_models = {}
    for idx, model_load_path in enumerate(model_load_paths):
      if idx == 0:
        continue
      sparsity = model_sparsities[idx]
      model = get_model(num_classes=int(num_class), path=model_load_path)
      model.eval()
      pruned_models[sparsity] = model
    # Dense -> all pruned in one pass over testloader
    dense_to_all_results = evaluate_dense_to_many_pruned_streaming(
        dense_model=dense_model,
        pruned_models_dict=pruned_models,
        testloader=testloader,
        epsilon=epsilon,
        device=device,
        mean=CIFAR_MEAN,
        std=CIFAR_STD
    )
    for idx, sparsity in enumerate(model_sparsities[1:]):
        transfer_d_p_lst[idx].append(dense_to_all_results[sparsity]['target_transfer_fooling_rate'])
        adv_acc_d_p_lst[idx].append(dense_to_all_results[sparsity]['target_adv_acc'])

        print(f"\nDense -> Pruned | Sparsity: {sparsity}, Epsilon: {epsilon}")
        print(f"Target transfer fooling rate: {dense_to_all_results[sparsity]['target_transfer_fooling_rate']}")

    # Pruned -> Dense still one source at a time
    for idx,sparsity in enumerate(model_sparsities[1:]):
        print(f"\nPruned -> Dense | Sparsity: {sparsity}, Epsilon: {epsilon}")

        pruned_to_dense = evaluate_transfer_attack(
            source_model=pruned_models[sparsity],
            target_model=dense_model,
            testloader=testloader,
            epsilon=epsilon,
            device=device,
            save_per_batch=0,
            mean=CIFAR_MEAN,
            std=CIFAR_STD
        )
        print(f"\nPruned -> Dense")
        print(f"Target transfer fooling rate: {pruned_to_dense['target_transfer_fooling_rate']}")

        transfer_p_d_lst[idx].append(pruned_to_dense['target_transfer_fooling_rate'])
        adv_acc_p_d_lst[idx].append(pruned_to_dense['target_adv_acc'])

    # Cleanup
    for model in pruned_models.values():
        del model
    del pruned_models
    torch.cuda.empty_cache()
  all_result[0]['data'] = transfer_d_p_lst
  all_result[1]['data'] = transfer_p_d_lst
  all_result[2]['data'] = adv_acc_d_p_lst
  all_result[3]['data'] = adv_acc_p_d_lst

  with open(f'/content/drive/MyDrive/DLBA_Project/Resnet_Result/Resnet18_T1_C{num_class}_result.json', 'a') as f:
    json.dump(all_result, f)

CIFAR-100 dataset downloaded and loaded successfully.
Loading model from path /content/drive/MyDrive/DLBA_Project/Models/finetuned_resnet18_cifar100_base_model.pth

================ Epsilon: 0.00784313725490196 ================

Loading model from path /content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar100_20.pth
Loading model from path /content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar100_40.pth
Loading model from path /content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar100_60.pth
Loading model from path /content/drive/MyDrive/DLBA_Project/Models/pruned_resnet18_cifar100_80.pth
[Dense->Many] Batch 0
[Dense->Many] Batch 20
[Dense->Many] Batch 40
[Dense->Many] Batch 60
[Dense->Many] Batch 80
[Dense->Many] Batch 100
[Dense->Many] Batch 120
[Dense->Many] Batch 140

Dense -> Pruned | Sparsity: 20, Epsilon: 0.00784313725490196
Target transfer fooling rate: 0.8857142857142857

Dense -> Pruned | Sparsity: 40, Epsilon: 0.00784313725490196
Target transfer 

In [27]:
print(all_result)

[{'title': 'Transferability: Dense → Pruned:', 'headers': ['Pruned Sparsity', '0.0078', '0.0156', '0.0313', '0.0627'], 'data': [['20%', 0.8857142857142857, 0.8275464445868034, 0.8225496476617553, 0.8736707238949392], ['40%', 0.8864802462801437, 0.8311954848640328, 0.8219599794766547, 0.8687788609543355], ['60%', 0.8581220900155199, 0.8158303155716503, 0.813502327987584, 0.8759699948266942], ['80%', 0.762982689747004, 0.7599201065246338, 0.8106524633821571, 0.8782956058588549]]}, {'title': 'Transferability: Pruned → Dense:', 'headers': ['Pruned Sparsity', '0.0078', '0.0156', '0.0313', '0.0627'], 'data': [['20%', 0.8893055735160998, 0.8405534721324195, 0.8313720418983577, 0.8810293547135652], ['40%', 0.8845208845208845, 0.8348635717056769, 0.8268459847407216, 0.8799948273632484], ['60%', 0.8602094917884392, 0.8154661838872366, 0.8229665071770335, 0.8823225139014613], ['80%', 0.7861114703219967, 0.7786111470321997, 0.809776283460494, 0.880124143282038]]}, {'title': 'Target adversarial acc

In [28]:
with open(f'/content/drive/MyDrive/DLBA_Project/Resnet_Result/Resnet18_T1_C100_result.json', 'a') as f:
  json.dump(all_result, f)